# Agentic Mesh | Agent Infrastructure

In [1]:
# Simulated Agentic Mesh with Service Discovery
# Full mesh enables peer-to-peer agent communication, not just orchestrator-mediated invocation
import time
from collections import defaultdict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import TypedDict, Dict, List
from typing_extensions import NotRequired

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Agent Registry -- simulates a distributed service registry
AGENT_REGISTRY: Dict[str, dict] = {}

def register_agent(name: str, capabilities: List[str], handler):
    """Register an agent with its capabilities."""
    AGENT_REGISTRY[name] = {
        "capabilities": capabilities,
        "handler": handler,
        "status": "healthy",
        "invocation_count": 0,
    }

def discover_agent(capability: str) -> str:
    """Find an agent that has the requested capability."""
    for name, info in AGENT_REGISTRY.items():
        if capability in info["capabilities"] and info["status"] == "healthy":
            return name
    return ""

def invoke_agent(name: str, payload: str) -> str:
    """Invoke an agent through the mesh (with tracking)."""
    agent = AGENT_REGISTRY.get(name)
    if not agent:
        return f"Error: Agent '{name}' not found in registry."
    agent["invocation_count"] += 1
    return agent["handler"](payload)

# Mesh feature: resilient invocation with retry + latency tracking
MESH_METRICS: Dict[str, dict] = defaultdict(lambda: {"latency_ms": [], "errors": 0})

def invoke_with_retry(capability: str, payload: str, max_retries: int = 2) -> str:
    """Discover and invoke with retry logic and latency tracking."""
    agent_name = discover_agent(capability)
    if not agent_name:
        return f"No agent found for capability: {capability}"
    for attempt in range(max_retries + 1):
        try:
            start = time.time()
            result = invoke_agent(agent_name, payload)
            latency = (time.time() - start) * 1000
            MESH_METRICS[agent_name]["latency_ms"].append(latency)
            return result
        except Exception as e:
            MESH_METRICS[agent_name]["errors"] += 1
            if attempt == max_retries:
                return f"Error after {max_retries + 1} attempts: {e}"
    return ""

In [4]:
# Define agent handlers
def data_agent_handler(payload: str) -> str:
    response = model.invoke(f"You are a data analysis agent. Analyze: {payload}")
    return response.content

def writer_agent_handler(payload: str) -> str:
    response = model.invoke(f"You are a report writing agent. Write a report based on: {payload}")
    return response.content

def summary_agent_handler(payload: str) -> str:
    response = model.invoke(f"You are a summarization agent. Summarize: {payload}")
    return response.content

# Register agents
register_agent("data_analyzer", ["data_analysis", "statistics", "trends"], data_agent_handler)
register_agent("report_writer", ["writing", "reports", "formatting"], writer_agent_handler)
register_agent("summarizer", ["summarization", "compression", "key_points"], summary_agent_handler)

In [5]:
class MeshState(TypedDict):
    task: str
    analysis: NotRequired[str]
    report: NotRequired[str]
    summary: NotRequired[str]

def orchestrate_via_mesh(state: MeshState) -> dict:
    """Orchestrator discovers and invokes agents through the mesh."""
    # Step 1: Discover and invoke data analysis agent
    analyst = discover_agent("data_analysis")
    analysis = invoke_agent(analyst, state["task"]) if analyst else "No analyst available."

    # Step 2: Discover and invoke report writer
    writer = discover_agent("writing")
    report = invoke_agent(writer, analysis) if writer else "No writer available."

    # Step 3: Discover and invoke summarizer
    summarizer = discover_agent("summarization")
    summary = invoke_agent(summarizer, report) if summarizer else "No summarizer available."

    return {"analysis": analysis, "report": report, "summary": summary}

graph = StateGraph(MeshState)
graph.add_node("orchestrate", orchestrate_via_mesh)
graph.add_edge(START, "orchestrate")
graph.add_edge("orchestrate", END)

mesh = graph.compile()
result = mesh.invoke({"task": "Analyze the trends in renewable energy adoption across G7 countries from 2020-2024"})
print(f"Summary:\n{result['summary']}")

# Show mesh telemetry (registry + mesh observability)
for name, info in AGENT_REGISTRY.items():
    metrics = MESH_METRICS.get(name, {"latency_ms": [], "errors": 0})
    avg_lat = sum(metrics["latency_ms"]) / max(len(metrics["latency_ms"]), 1)
    print(f"\nAgent '{name}': invoked {info['invocation_count']}x, avg latency: {avg_lat:.0f}ms, errors: {metrics['errors']}")

Summary:
The report on renewable energy adoption trends in G7 countries from 2020 to 2024 highlights significant growth driven by climate commitments and sustainability goals. Key elements include reinforced policies, such as subsidies and tax incentives in Germany, France, and the UK, fostering renewable energy growth. Technological progress, particularly in solar and wind energy, has lowered costs and improved grid integration. Investment in renewables has surged, with the public and private sectors supporting infrastructure development, especially in solar and wind. Country-specific strides include the US's solar and wind growth, Germany's ongoing Energiewende initiative, and Japan's investment in solar PV since the Fukushima disaster. Challenges include integrating renewables into grids and navigating economic and geopolitical factors. The outlook for 2024 points to continued growth, supported by investments, technological innovations, and policies aligned with international goals 